# Start of the Bronze process

This notebook initiates the transformation of the Bronze layer, which receives data already loaded in the CVS raw file, without applying any data processing.

- Definition of variables, paths, and catalog schema
- Configuration of credentials and secure access
- Writing the Bronze tables to the Data Lake container

In [0]:
from datetime import datetime as dt

tier = "bronze"
raw_data = 'olistdataraw'
storage_account = "databrickslrd"


spark.sql("CREATE SCHEMA IF NOT EXISTS " + tier)
spark.sql("USE " + tier)


access_key = dbutils.secrets.get(
    scope = "olist-scope", 
    key = "access-key"
    )

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    access_key
)

PATH_RAW = f'wasbs://{raw_data}@{storage_account}.blob.core.windows.net/archive'

PATH_STORAGE = f"wasbs://{tier}@{storage_account}.blob.core.windows.net"


archives = dbutils.fs.ls(PATH_RAW)

arch_name = [arch.name for arch in archives]

def create_bronze_table(path_table):
    cut_name = path_table.split('_dataset')[0][6:]
    df = spark.read.option("header", True).option("inferSchema", True).csv(f"{PATH_RAW}/{path_table}")

    df.write.format("delta").mode("overwrite").save(f"{PATH_STORAGE}/{cut_name}_{dt.now().strftime('%Y%m%d')}")

    df.write.format("delta").mode("overwrite").saveAsTable(cut_name)





In [0]:
for name in arch_name:
    if "translation" not in name:
        create_bronze_table(name)